In [3]:
# ============================================================
#   RISK-ONLY LOG-RETURN MODEL: BASELINE vs MSF-CAL (ATTENTION)
# ============================================================

import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, callbacks

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_recall_fscore_support

print("TF version:", tf.__version__)

# -------------------------
# 1. CONFIG
# -------------------------
START_DATE = "2010-01-01"
END_DATE   = datetime.today().strftime("%Y-%m-%d")

# A few large liquid Indian stocks + indices
STOCK_TICKERS = [
    "RELIANCE.NS",
    "TCS.NS",
    "INFY.NS",
    "HDFCBANK.NS",
    "ICICIBANK.NS"
]

INDEX_TICKERS = {
    "^NSEI":   "idx_nifty50",   # Nifty 50
    "^CNXIT":  "idx_nifty_it",  # Nifty IT
}

WINDOW_LENGTH = 60   # T (days in each input window)
FORECAST_HORIZON = 1 # predict next-day log return risk

BIG_MOVE_QUANTILE = 0.90  # define big move as top 10% |log_return|

# -------------------------
# 2. BASIC HELPERS
# -------------------------
def download_ohlcv(ticker, start=START_DATE, end=END_DATE):
    """
    Download OHLCV from yfinance and normalise columns.
    Handles both normal and MultiIndex columns.
    """
    print(f"Downloading {ticker} ...")
    df = yf.download(ticker, start=start, end=end, auto_adjust=False)

    # If columns are MultiIndex (e.g., ('Close','^NSEI')), flatten to first level
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    # Keep only needed OHLCV columns
    needed_cols = ["Open", "High", "Low", "Close", "Volume"]
    df = df[needed_cols].copy()
    df = df.dropna()
    print(f"  Shape after cleaning: {df.shape}")
    return df

def add_features(df):
    """
    Add basic features including log returns.
    Assumes df has columns: Open, High, Low, Close, Volume.
    """
    d = df.copy()
    d["log_ret"] = np.log(d["Close"]).diff()

    # Simple moving averages
    d["ma_5"]   = d["Close"].rolling(window=5).mean()
    d["ma_10"]  = d["Close"].rolling(window=10).mean()
    d["ma_20"]  = d["Close"].rolling(window=20).mean()

    # Volatility (rolling std of log returns)
    d["vol_5"]  = d["log_ret"].rolling(window=5).std()
    d["vol_10"] = d["log_ret"].rolling(window=10).std()

    # Volume features
    d["vol_ma_5"]  = d["Volume"].rolling(window=5).mean()
    d["vol_ma_10"] = d["Volume"].rolling(window=10).mean()

    # Drop initial NaNs from rolling
    d = d.dropna()
    return d

# -------------------------
# 3. INDEX FEATURES
# -------------------------
print("\n=== Downloading indices and building index features ===")
idx_frames = []
for ticker, prefix in INDEX_TICKERS.items():
    raw = download_ohlcv(ticker)
    feat = add_features(raw)

    # Only keep Close and log_ret for index, rename them
    idx_feat = feat[["Close", "log_ret"]].copy()
    idx_feat.columns = [f"{prefix}_close", f"{prefix}_log_ret"]
    idx_frames.append(idx_feat)

# Intersect all index date ranges
idx_all = idx_frames[0]
for f in idx_frames[1:]:
    idx_all = idx_all.join(f, how="inner")

print("Combined index features shape:", idx_all.shape)
print("Index feature columns:", idx_all.columns.tolist())

# -------------------------
# 4. STOCK FEATURES + JOIN INDEX FEATURES
# -------------------------
print("\n=== Downloading stocks and joining indices ===")

stock_dfs = []
for ticker in STOCK_TICKERS:
    raw = download_ohlcv(ticker)
    feat = add_features(raw)

    # join index features on date index (inner join)
    joined = feat.join(idx_all, how="inner")

    # target log return: next-day move of the stock itself
    joined["log_ret_target"] = joined["log_ret"].shift(-FORECAST_HORIZON)

    # Drop rows where we don't have next-day target
    joined = joined.dropna(subset=["log_ret_target"])

    joined["ticker"] = ticker
    stock_dfs.append(joined)

all_data = pd.concat(stock_dfs, axis=0).sort_index()
print("All combined stock+index data shape:", all_data.shape)
print("Columns:", all_data.columns.tolist())

# -------------------------
# 5. BUILD WINDOWS (X) + CONTINUOUS TARGET (z)
# -------------------------
feature_cols = [c for c in all_data.columns
                if c not in ["log_ret_target", "ticker"]]

def make_windows_for_one_stock(df_stock, ticker, T=WINDOW_LENGTH):
    """
    df_stock: dataframe for single ticker, already sorted by date and
              containing feature_cols + log_ret_target
    """
    df_stock = df_stock.sort_index()
    values   = df_stock[feature_cols].values
    z_target = df_stock["log_ret_target"].values
    dates    = df_stock.index.values

    X_list, z_list, date_list = [], [], []
    for i in range(T-1, len(df_stock)):
        x_win = values[i-T+1:i+1]
        z     = z_target[i]
        if np.isnan(z):
            continue
        X_list.append(x_win)
        z_list.append(z)
        date_list.append(dates[i])

    X = np.array(X_list)
    z = np.array(z_list)
    dates = np.array(date_list)
    tickers = np.array([ticker] * len(X))
    return X, z, dates, tickers

X_all_list, z_all_list, date_all_list, ticker_all_list = [], [], [], []

for ticker in STOCK_TICKERS:
    df_t = all_data[all_data["ticker"] == ticker]
    X_t, z_t, d_t, tick_t = make_windows_for_one_stock(df_t, ticker, T=WINDOW_LENGTH)
    X_all_list.append(X_t)
    z_all_list.append(z_t)
    date_all_list.append(d_t)
    ticker_all_list.append(tick_t)

X_all = np.concatenate(X_all_list, axis=0)
z_all = np.concatenate(z_all_list, axis=0)
dates_all = np.concatenate(date_all_list, axis=0)
tickers_all = np.concatenate(ticker_all_list, axis=0)

print("\nWindows built:")
print("X_all shape:", X_all.shape)   # (N, T, F)
print("z_all shape:", z_all.shape)   # (N,)
print("Example dates range:", dates_all.min(), "to", dates_all.max())

# -------------------------
# 6. SORT BY DATE & TRAIN/VAL/TEST SPLIT
# -------------------------
order = np.argsort(dates_all)
X_all   = X_all[order]
z_all   = z_all[order]
dates_all   = dates_all[order]
tickers_all = tickers_all[order]

N = len(z_all)
train_end = int(0.7 * N)
val_end   = int(0.85 * N)

X_train, X_val, X_test = X_all[:train_end], X_all[train_end:val_end], X_all[val_end:]
z_train, z_val, z_test = z_all[:train_end], z_all[train_end:val_end], z_all[val_end:]

print("\nSplit sizes:")
print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

# -------------------------
# 7. DEFINE BIG-MOVE THRESHOLD & BINARY LABELS
# -------------------------
abs_train = np.abs(z_train)
big_move_threshold = np.quantile(abs_train, BIG_MOVE_QUANTILE)
print(f"\nBig-move threshold based on train |log_ret| (q={BIG_MOVE_QUANTILE}):",
      big_move_threshold)

y_train = (np.abs(z_train) >= big_move_threshold).astype(int)
y_val   = (np.abs(z_val)   >= big_move_threshold).astype(int)
y_test  = (np.abs(z_test)  >= big_move_threshold).astype(int)

print("Positive rate (train):", y_train.mean())
print("Positive rate (val):  ", y_val.mean())
print("Positive rate (test): ", y_test.mean())

# -------------------------
# 8. STANDARDIZE FEATURES
# -------------------------
N_train, T, F = X_train.shape
print("\nWindow length T:", T, "Num features F:", F)

scaler = StandardScaler()
X_train_flat = X_train.reshape(-1, F)
X_val_flat   = X_val.reshape(-1, F)
X_test_flat  = X_test.reshape(-1, F)

scaler.fit(X_train_flat)
X_train_s = scaler.transform(X_train_flat).reshape(N_train, T, F)
X_val_s   = scaler.transform(X_val_flat).reshape(X_val.shape[0], T, F)
X_test_s  = scaler.transform(X_test_flat).reshape(X_test.shape[0], T, F)

# -------------------------
# 9. CLASS WEIGHTS
# -------------------------
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
neg = float(neg)
pos = float(pos)

w_pos = (neg + pos) / (2.0 * pos)
w_neg = (neg + pos) / (2.0 * neg)
class_weights = {0: w_neg, 1: w_pos}
print("\nClass weights:", class_weights)

# -------------------------
# 10. BASELINE RISK-ONLY MODEL
# -------------------------
def build_baseline_risk(T, F):
    inp = layers.Input(shape=(T, F), name="input")

    x = layers.Conv1D(32, 5, padding="same", activation="relu")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Bidirectional(layers.LSTM(32, return_sequences=False))(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.3)(x)

    out = layers.Dense(1, activation="sigmoid", name="risk_out")(x)

    model = models.Model(inputs=inp, outputs=out, name="BaselineRisk")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.BinaryAccuracy(name="accuracy")
        ]
    )
    return model

baseline = build_baseline_risk(T, F)
baseline.summary()

early_base = callbacks.EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=6,
    restore_best_weights=True
)

reduce_base = callbacks.ReduceLROnPlateau(
    monitor="val_auc",
    mode="max",
    factor=0.5,
    patience=3,
    verbose=1
)

print("\n================ TRAINING BASELINE (risk only) ================\n")
hist_base = baseline.fit(
    X_train_s,
    y_train,
    validation_data=(X_val_s, y_val),
    epochs=60,
    batch_size=256,
    class_weight=class_weights,
    callbacks=[early_base, reduce_base],
    verbose=1
)

# -------------------------
# 11. MSF-CAL RISK-ONLY MODEL (multi-scale + attention)
# -------------------------
def temporal_attention_block(H, att_units=32, dropout_rate=0.2):
    """
    H: (batch, T, H_dim), attention over the time axis.
    Returns:
      - context: (batch, H_dim)
      - att_weights: (batch, T)
    All TF ops are wrapped in Keras Lambda layers to avoid KerasTensor errors.
    """
    # 1) Compute raw attention scores e_t
    score = layers.TimeDistributed(
        layers.Dense(att_units, activation="tanh")
    )(H)  # (batch, T, att_units)

    score = layers.TimeDistributed(
        layers.Dense(1, activation=None)
    )(score)  # (batch, T, 1)

    # 2) Squeeze last dim -> (batch, T)
    score = layers.Lambda(lambda x: tf.squeeze(x, axis=-1))(score)

    # 3) Softmax over time -> attention weights alpha_t
    att_weights = layers.Activation("softmax", name="att_weights")(score)  # (batch, T)

    # 4) Expand back to (batch, T, 1) so we can multiply with H
    att_weights_expanded = layers.Lambda(
        lambda x: tf.expand_dims(x, axis=-1)
    )(att_weights)  # (batch, T, 1)

    # 5) Weighted sum over time axis
    weighted_H = layers.Multiply()([H, att_weights_expanded])  # (batch, T, H_dim)
    context = layers.Lambda(lambda x: tf.reduce_sum(x, axis=1))(weighted_H)  # (batch, H_dim)

    # 6) Optional dropout on context
    context = layers.Dropout(dropout_rate)(context)

    return context, att_weights


def build_msfcal_risk(T, F):
    inp = layers.Input(shape=(T, F), name="input")
    reg = regularizers.l2(1e-4)

    # Multi-scale CNN branches
    c3 = layers.Conv1D(16, 3, padding="same", activation="relu",
                       kernel_regularizer=reg)(inp)
    c5 = layers.Conv1D(16, 5, padding="same", activation="relu",
                       kernel_regularizer=reg)(inp)
    c7 = layers.Conv1D(16, 7, padding="same", activation="relu",
                       kernel_regularizer=reg)(inp)

    x = layers.Concatenate(axis=-1)([c3, c5, c7])   # (batch, T, 48)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Bidirectional(
        layers.LSTM(32, return_sequences=True, kernel_regularizer=reg)
    )(x)
    x = layers.Dropout(0.3)(x)

    context, att_weights = temporal_attention_block(x, att_units=32, dropout_rate=0.3)

    h = layers.Dense(32, activation="relu", kernel_regularizer=reg)(context)
    h = layers.Dropout(0.4)(h)
    out = layers.Dense(1, activation="sigmoid", name="risk_out")(h)

    model = models.Model(inputs=inp, outputs=out, name="MSF_CAL_Risk")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.BinaryAccuracy(name="accuracy")
        ]
    )
    return model

msf_cal = build_msfcal_risk(T, F)
msf_cal.summary()

early_msf = callbacks.EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=8,
    restore_best_weights=True
)

reduce_msf = callbacks.ReduceLROnPlateau(
    monitor="val_auc",
    mode="max",
    factor=0.5,
    patience=4,
    verbose=1
)

print("\n================ TRAINING MSF-CAL (risk only, att-regularized) ================\n")
hist_msf = msf_cal.fit(
    X_train_s,
    y_train,
    validation_data=(X_val_s, y_val),
    epochs=60,
    batch_size=256,
    class_weight=class_weights,
    callbacks=[early_msf, reduce_msf],
    verbose=1
)

# -------------------------
# 12. EVALUATION HELPERS
# -------------------------
def eval_model(name, model, X_test, y_test, k_frac=0.10):
    print(f"\n=== {name} TEST EVAL ===")
    proba = model.predict(X_test, verbose=0).ravel()

    # Threshold=0.5
    y_pred = (proba >= 0.5).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="binary", zero_division=0
    )
    print(f"Threshold=0.5 -> precision: {prec:.3f}, recall: {rec:.3f}, F1: {f1:.3f}")

    # Top-k (e.g. top 10%) most risky days
    n = len(proba)
    k = int(np.round(k_frac * n))
    idx_sorted = np.argsort(-proba)  # descending
    top_idx = idx_sorted[:k]
    y_top = y_test[top_idx]

    prec_top = y_top.mean()
    rec_top = y_top.sum() / y_test.sum()
    f1_top = 2 * prec_top * rec_top / (prec_top + rec_top + 1e-8)
    print(f"Top {k_frac*100:.0f}% days -> precision: {prec_top:.3f}, "
          f"recall: {rec_top:.3f}, F1: {f1_top:.3f}")

    return proba

# -------------------------
# 13. FINAL TEST EVAL
# -------------------------
proba_base = eval_model("Baseline", baseline, X_test_s, y_test, k_frac=0.10)
proba_msf  = eval_model("MSF-CAL", msf_cal, X_test_s, y_test, k_frac=0.10)

print("\nBig-move threshold used (for labels): |log_ret| >=", big_move_threshold)
print("Test positives:", y_test.sum(), "out of", len(y_test))


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

TF version: 2.19.0

=== Downloading indices and building index features ===
  Shape after cleaning: (3945, 5)
  Shape after cleaning: (3654, 5)
Combined index features shape: (3633, 4)
Index feature columns: ['idx_nifty50_close', 'idx_nifty50_log_ret', 'idx_nifty_it_close', 'idx_nifty_it_log_ret']

=== Downloading stocks and joining indices ===



[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


  Shape after cleaning: (3968, 5)
  Shape after cleaning: (3968, 5)


[*********************100%***********************]  1 of 1 completed


...

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 60, 17)    │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 60, 16)    │        832 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_6 (Conv1D)   │ (None, 60, 16)    │      1,376 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_7 (Conv1D)   │ (None, 60, 16)    │      1,920 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 60, 48)    │          0 │ conv1d_5[0][0],   │
│ (Concatenate)       │                   │            │ conv1d_6[0][0],   │
│                     │                   │            │ conv1d_7[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 60, 48)    │        192 │ concatenate_1[0]… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_8 (Dropout) │ (None, 60, 48)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_3     │ (None, 60, 64)    │     20,736 │ dropout_8[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_9 (Dropout) │ (None, 60, 64)    │          0 │ bidirectional_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_2  │ (None, 60, 32)    │      2,080 │ dropout_9[0][0]   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_3  │ (None, 60, 1)     │         33 │ time_distributed… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 60)        │          0 │ time_distributed… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ att_weights         │ (None, 60)        │          0 │ lambda[0][0]      │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 60, 1)     │          0 │ att_weights[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 60, 64)    │          0 │ dropout_9[0][0],  │
│                     │                   │            │ lambda_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_2 (Lambda)   │ (None, 64)        │          0 │ multiply[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_10          │ (None, 64)        │          0 │ lambda_2[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 32)        │      2,080 │ dropout_10[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_11          │ (None, 32)        │          0 │ dense_6[0][0]     │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ risk_out (Dense)    │ (None, 1)         │         33 │ dropout_11[0][0]

 Total params: 29,282 (114.38 KB)

 Trainable params: 29,186 (114.01 KB)

 Non-trainable params: 96 (384.00 B)


================ TRAINING MSF-CAL (risk only, att-regularized) ================

Epoch 1/60
49/49 ━━━━━━━━━━━━━━━━━━━━ 34s 420ms/step - accuracy: 0.5456 - auc: 0.5686 - loss: 0.7307 - val_accuracy: 0.9153 - val_auc: 0.5443 - val_loss: 0.5544 - learning_rate: 0.0010
Epoch 2/60
49/49 ━━━━━━━━━━━━━━━━━━━━ 18s 373ms/step - accuracy: 0.6087 - auc: 0.6595 - loss: 0.6840 - val_accuracy: 0.8399 - val_auc: 0.5663 - val_loss: 0.5810 - learning_rate: 0.0010
Epoch 3/60
49/49 ━━━━━━━━━━━━━━━━━━━━ 18s 363ms/step - accuracy: 0.5999 - auc: 0.6532 - loss: 0.6858 - val_accuracy: 0.9071 - val_auc: 0.6016 - val_loss: 0.4793 - learning_rate: 0.0010
Epoch 4/60
49/49 ━━━━━━━━━━━━━━━━━━━━ 17s 356ms/step - accuracy: 0.6109 - auc: 0.6744 - loss: 0.6643 - val_accuracy: 0.9429 - val_auc: 0.5826 - val_loss: 0.5105 - learning_rate: 0.0010
Epoch 5/60
49/49 ━━━━━━━━━━━━━━━━━━━━ 21s 362ms/step - accuracy: 0.6143 - auc: 0.6798 - loss: 0.6565 - val_accuracy: 0.9429 - val_auc: 0.6102 - val_loss: 0.4852 - learning_rate: 